# 임베딩 실습 보충 노트북

「임베딩_실습_손코딩」 파일을 치면서 막히는 부분을 보충합니다.

- **1절 · 2절**: 코사인 유사도를 numpy 로 직접 계산해 봅니다. 모델이 없어도 실행됩니다.
- **3절**: 단어 데이터를 실습 파일과 같은 방식으로 점검합니다.
- **4절**: 토크나이저만 불러와 단어가 어떻게 쪼개지는지 봅니다. 모델 전체를 받지 않아 빠릅니다.
- **5절**: 자주 나는 오류와 해결 방법입니다.
- **6절**: 내 데이터로 해 보는 연습 과제입니다.

## 1. 코사인 유사도를 직접 계산하기

코사인 유사도 = 내적 ÷ (벡터 길이 × 벡터 길이) 입니다. 먼저 2차원 벡터로 손으로 계산한 값과 비교합니다.

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

a = np.array([3.0, 4.0])
b = np.array([4.0, 3.0])

# 1) 공식 그대로 계산
manual = a @ b / (np.linalg.norm(a) * np.linalg.norm(b))
print("공식으로 계산:", round(manual, 3))

# 2) 길이를 1로 맞춘 뒤 내적
an, bn = a / np.linalg.norm(a), b / np.linalg.norm(b)
print("정규화 후 내적:", round(an @ bn, 3))

# 3) sklearn 과 비교
print("sklearn:", round(cosine_similarity([a], [b])[0, 0], 3))

1024차원 임의의 벡터 5개로도 확인합니다. 정규화한 뒤 내적한 결과가 sklearn 과 같으면 실습 파일의 `emb @ emb.T` 가 코사인 행렬이라는 뜻입니다.

In [ ]:
rng = np.random.default_rng(0)
X = rng.normal(size=(5, 1024))
Xn = X / np.linalg.norm(X, axis=1, keepdims=True)

print("정규화 후 내적 == sklearn:", np.allclose(Xn @ Xn.T, cosine_similarity(X)))

## 2. 정규화를 하지 않으면 무슨 일이 생기나

내적은 벡터 길이에 따라 값이 달라지지만, 코사인은 방향만 봅니다. 같은 방향의 벡터를 10배 길게 만들어 확인합니다.

In [ ]:
v = np.array([1.0, 2.0])

print("길이 그대로  → 내적:", v @ v, "/ 코사인:", cosine_similarity([v], [v])[0, 0].round(3))
print("길이 10배    → 내적:", (10 * v) @ v, "/ 코사인:", cosine_similarity([10 * v], [v])[0, 0].round(3))

- 내적은 10배가 됐지만 코사인은 1.0 그대로입니다.
- 그래서 실습 파일에서 `normalize_embeddings=True` 를 빼먹으면 유사도 값이 길이에 따라 흔들립니다.

## 3. 단어 데이터 점검

실습 파일의 2절 데이터와 같은 내용입니다. 같은 단어가 두 번 들어가거나 범주 크기가 다르면 군집 결과가 이상해집니다. 아래 셀로 먼저 확인합니다.

In [ ]:
words_by_group = {
    "음식": ["피자", "김밥", "라면", "떡볶이", "비빔밥", "샐러드"],
    "색깔": ["빨강", "파랑", "노랑", "초록", "보라", "검정"],
    "악기": ["피아노", "기타", "바이올린", "드럼", "플루트", "첼로"],
    "감정": ["기쁨", "슬픔", "화남", "두려움", "놀람", "설렘"],
    "학교": ["학생", "선생님", "교실", "칠판", "책상", "운동장"],
}
words = [w for ws in words_by_group.values() for w in ws]

assert len(words) == 30, "단어는 30개여야 합니다"
assert len(set(words)) == 30, "중복된 단어가 있습니다"
assert all(len(ws) == 6 for ws in words_by_group.values()), "범주마다 6개씩이어야 합니다"
print("점검 통과:", len(words), "개 단어, 중복 없음")

## 4. 토큰 쪼개기 미리 보기

토크나이저만 불러와 단어가 어떤 토큰으로 나뉘는지 봅니다. 실습 파일 「3. 최근접 이웃 검색」에서 「피자」와 「피아노」가 가까운 이유를 여기서 확인할 수 있습니다.

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("nlpai-lab/KURE-v1")
for w in ["피자", "피아노", "기쁨", "선생님", "바이올린"]:
    print(w, "→", tok.tokenize(w))

## 5. 자주 나는 오류

| 증상 | 원인 | 해결 |
|---|---|---|
| `NameError: name 'emb' is not defined` | 앞의 셀을 건너뛰었음 | 런타임 → 세션 다시 시작 후 셀을 처음부터 차례로 실행 |
| `ModuleNotFoundError: koreanize_matplotlib` | 설치 셀을 실행하지 않았음 | 「1. 준비」의 `!pip` 셀을 실행한 뒤 런타임을 다시 시작 |
| 그림의 한글이 네모로 보임 | `koreanize_matplotlib` 가 적용되지 않음 | import 셀을 다시 실행하고 그림 셀을 다시 실행 |
| `You are sending unauthenticated requests to the HF Hub` | Hugging Face 토큰이 없음 | 무시해도 됩니다. 다운로드는 계속 진행됩니다 |
| `RuntimeWarning: divide by zero` (UMAP) | UMAP 내부 계산의 경고 | 결과에 영향이 없으면 무시해도 됩니다 |
| 모델 다운로드가 1분 넘게 멈춤 | 네트워크 또는 Colab 상태 | 기다린 뒤 해당 셀을 다시 실행 |

## 6. 연습 과제 (내 데이터)

아래 빈 셀에 직접 써서 실행해 봅니다. 실습 파일의 셀을 복사해 고쳐 써도 됩니다.

In [ ]:
# 연습 1: 내 주제의 단어 5개 범주 × 6개로 바꿔 보기
# 힌트: words_by_group 을 바꾸고, 실습 파일의 2절부터 6절까지 다시 실행합니다.


In [ ]:
# 연습 2: 뜻이 둘인 단어 「배」(먹는 배 · 타는 배)의 이웃 찾기
# 힌트: model.encode([...], normalize_embeddings=True) 로 벡터를 만든 뒤, 실습 파일의 nn.kneighbors(...) 로 이웃을 찾습니다.


In [ ]:
# 연습 3: 뜻이 반대인 두 문장의 코사인 구하기
# 예: 「배송이 빨라요」, 「배송이 너무 느려요」
# 힌트: 문장 벡터끼리 내적(@)을 구하면 코사인입니다.
